Import necessary libraries and CSV data files

In [349]:
#Import libraries

import pandas as pd
import sqlite3

#Read each csv and create a corresponding pandas dataframe.

sales_data = pd.read_csv("Video_Game_Sales_1978-2024.csv")
top_50_data = pd.read_csv("2024_Top_50_AAA_AA_Indie_Games.csv")
gaming_study_data = pd.read_csv("GamingStudy_data.csv", encoding='windows-1253')
online_behavior_data = pd.read_csv("online_gaming_behavior_dataset.csv")
genres = pd.read_csv('genres.csv')



In [350]:
#Let's examine our initial data.  

print(sales_data.head(), '\n')
print(sales_data.info(), '\n')

print(top_50_data.head(), '\n')
print(top_50_data.info(), '\n')

print(gaming_study_data.head(), '\n')
print(gaming_study_data.info(), '\n')

print(online_behavior_data.head(), '\n')
print(online_behavior_data.info(), '\n')

print(genres.head(), '\n')
print(genres.info())

   Rank              Name Platform All_Platforms  \
0     1            Tetris   Series           NaN   
1     2           Pokemon   Series           NaN   
2     3      Call of Duty   Series           NaN   
3     4  Grand Theft Auto   Series           NaN   
4     5       Super Mario   Series           NaN   

                                           All_Games             Publisher  \
0  Tetris (1984)|Tetris (1989)|Welltris|Hatris|Tw...  The Tetris Company     
1  Pokemon Red & Green (Japan-only) & Blue|Pokemo...            Nintendo     
2  Call of Duty|Call of Duty 2|Call of Duty 3|Cal...          Activision     
3  Grand Theft Auto|Grand Theft Auto: London 1969...      Rockstar Games     
4  Mario Bros.|Super Mario Bros.|Super Mario Bros...            Nintendo     

           Developer  Critic_Score  User_Score  NA_Sales  PAL_Sales  JP_Sales  \
0  Alexey Pajitnov             NaN         NaN       NaN        NaN       NaN   
1       Game Freak             NaN         NaN       NaN

In [351]:
#Add normalization of genre data to top_50_data
top_50_data = pd.merge(top_50_data, genres[['Name', 'Genre']], on='Name', how='left')

#Align column names and rename columns for clarity.

top_50_data.rename(columns={"ReleaseDate": "Year"}, inplace=True)
gaming_study_data.rename(columns={"Game": "Name", "Hours": "HoursPerWeek", "Residence_ISO3": "Location", "GAD_T": "GAD_Total", "SWL_T": "SWL_Total", "SPIN_T": "SPIN_Total"}, inplace=True)
online_behavior_data.rename(columns={"GameGenre": "Genre", "PlayTimeHours": "HoursPerWeek"}, inplace=True)



In [352]:
#Drop unnecessary columns from dataframe.

sales_data = sales_data.drop(['Rank', 'Publisher', 'Developer', 'Critic_Score', 'All_Platforms', 'User_Score', 'All_Games', 'NA_Sales', 'PAL_Sales', 'JP_Sales', 'Other_Sales'], axis=1)
top_50_data = top_50_data.drop(['Publishers', 'Developers', 'Steam Id', 'Review Count', 'Review Score', 'Steam Followers'], axis=1)
top_50_data = top_50_data.drop(top_50_data.columns[0], axis=1)
gaming_study_data = gaming_study_data.drop(['S. No.', 'Timestamp', 'GADE', 'earnings', 'whyplay', 'Degree', 'Playstyle', 'Work', 'League', 'highestleague', 'streams', 'Birthplace', 'Residence', 'Reference', 'accept', 'Birthplace_ISO3'], axis=1)
online_behavior_data = online_behavior_data.drop(['PlayerID', 'PlayerLevel', 'AchievementsUnlocked'], axis=1)

print(sales_data.head())
print(top_50_data.head())
print(gaming_study_data.head())
print(online_behavior_data.head())

               Name Platform  Global_Sales    Year             Genre
0            Tetris   Series           NaN  1988.0            Puzzle
1           Pokemon   Series           NaN  1998.0      Role-Playing
2      Call of Duty   Series           NaN  2003.0           Shooter
3  Grand Theft Auto   Series           NaN  1998.0  Action-Adventure
4       Super Mario   Series           NaN  1983.0          Platform
                               Name       Year   Price    Copies Sold  \
0                Black Myth: Wukong  8/19/2024  $59.99  21,836,817.00   
1                      HELLDIVERS 2   2/8/2024  $39.99  12,410,685.00   
2                          Palworld  1/18/2024  $29.99  18,567,241.00   
3  Warhammer 40,000: Space Marine 2   9/9/2024  $59.99   2,709,725.00   
4                   Path of Exile 2  12/6/2024  $29.99   5,015,358.00   

     Gross Revenue  Average Playtime (Hrs) PublisherClass  Early Access  \
0  $951,505,160.00                    60.4            AAA         False 

In [353]:
#Remove rows with NA for NAME
top_50_data = top_50_data.dropna(subset='Name')
sales_data = sales_data.dropna(subset='Name')
gaming_study_data = gaming_study_data.dropna(subset='Name')

In [ ]:
#Remove all game series from sales data as well as cross-platform entries as these have no sales

sales_data = sales_data.drop(sales_data[(sales_data['Platform'] == ('Series'))].index)
sales_data = sales_data.dropna(subset="Global_Sales")

print(sales_data.info())

#Convert any necessary columns to correct data type
sales_data['Name'] = sales_data['Name'].astype(str)
sales_data['Genre'] = sales_data['Genre'].astype(str)

#Update PokÃ©mon to Pokemon in the sales data. As there are various versions of 'Starcraft II' on the sales data sheet this is updated to 'Starcraft' to match the gaming study data.

sales_data['Name'] = sales_data['Name'].str.replace('PokÃ©mon', 'Pokemon')
sales_data.loc[sales_data['Name'].str.contains('StarCraft II'), 'Name'] = 'Starcraft 2'

online_behavior_data['Genre'] = online_behavior_data['Genre'].replace('RPG', 'Role-Playing')

<class 'pandas.core.frame.DataFrame'>
Index: 20301 entries, 29 to 23357
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Name          20301 non-null  object 
 1   Platform      20301 non-null  object 
 2   Global_Sales  20301 non-null  float64
 3   Year          20217 non-null  float64
 4   Genre         20301 non-null  object 
dtypes: float64(2), object(3)
memory usage: 951.6+ KB
None


In [355]:
#Remove rows with NA for SPIN
gaming_study_data = gaming_study_data.dropna(subset='SPIN_Total')

#Clean up the Platform data - only PC and console are differentiated so no need for the special characters. Similarly Smartphone / Tablet can be consolidated as Mobile.
gaming_study_data['Platform'] = gaming_study_data['Platform'].replace('Console (PS, Xbox, ...)', 'Console')
gaming_study_data['Platform'] = gaming_study_data['Platform'].replace('Smartphone / Tablet', 'Mobile')

#The names of the games on the gaming study data should match those on the sales data sheet as well.

gaming_study_data['Name'] = gaming_study_data['Name'].replace('Skyrim', 'The Elder Scrolls V: Skyrim')
gaming_study_data['Name'] = gaming_study_data['Name'].replace('Counter Strike', 'Counter-Strike')
gaming_study_data['Name'] = gaming_study_data['Name'].replace('Diablo 3', 'Diablo III')

#Clean up the Platform data - only PC and console are differentiated so no need for the special characters. Similarly Smartphone / Tablet can be consolidated as Mobile.
gaming_study_data['Platform'] = gaming_study_data['Platform'].replace('Console (PS, Xbox, ...)', 'Console')
gaming_study_data['Platform'] = gaming_study_data['Platform'].replace('Smartphone / Tablet', 'Mobile')

#The names of the games on the gaming study data should match those on the sales data sheet as well.

gaming_study_data['Name'] = gaming_study_data['Name'].replace('Skyrim', 'The Elder Scrolls V: Skyrim')
gaming_study_data['Name'] = gaming_study_data['Name'].replace('Counter Strike', 'Counter-Strike')
gaming_study_data['Name'] = gaming_study_data['Name'].replace('Diablo 3', 'Diablo III')

print(gaming_study_data.head())

   GAD1  GAD2  GAD3  GAD4  GAD5  GAD6  GAD7  SWL1  SWL2  SWL3  ...  SPIN15  \
0     0     0     0     0     1     0     0     3     5     5  ...     0.0   
1     1     2     2     2     0     1     0     3     5     2  ...     3.0   
2     0     2     2     0     0     3     1     2     6     5  ...     4.0   
3     0     0     0     0     0     0     0     2     5     5  ...     1.0   
4     2     1     2     2     2     3     2     2     2     4  ...     0.0   

   SPIN16 SPIN17 Narcissism  Gender  Age  GAD_Total  SWL_Total  SPIN_Total  \
0     1.0    0.0        1.0    Male   25          1         23         5.0   
1     1.0    2.0        1.0    Male   41          8         16        33.0   
2     4.0    2.0        4.0  Female   32          8         17        31.0   
3     0.0    0.0        2.0    Male   28          0         17        11.0   
4     3.0    0.0        1.0    Male   19         14         14        13.0   

   Location  
0       USA  
1       USA  
2       DEU  
3     

In [356]:
#Utility function to get average value of one specified column based on given value in another column (defaults to Name).

def get_average(temp_df, tlist: list[str], cname: str, kname: str='Name'):
    """Calculates average value of specified column based on given value in another column (defaults to Name).
    Inputs:  Pandas dataframe, list of unique values for key column, name of column to calculate, optional key column specification.
    Outputs:  No return value; appends "cname_avg" column to temp_df."""
    for i in range(len(tlist)):
        mask = temp_df[kname] == tlist[i]
        average_val = temp_df[temp_df[kname] == tlist[i]][cname].mean()
        temp_df.loc[temp_df[kname] == tlist[i], [cname+'_avg']]= average_val

In [357]:
def duplist(temp_df):
    """Provides a list of values in the Name field  of the sales_data that appear more than once.  Only exact matches are returned.
    For example, 'Tetris', 'Tetris DS', and 'Tetris Plus' are all considered unique values."""
    multi_series = temp_df['Name'].value_counts()
    multi_list = multi_series[multi_series > 1].index.to_list()
    multi_list = list(set(multi_list))
    return multi_list

In [358]:
#If there is a title with multiple rows but no row where the Platform value is All, create a new row for the title.
#Genre and Year values will be based taken from the earliest release year.

mask = sales_data['Name'].isin(duplist(sales_data))
placeholder = sales_data[mask] #dataframe with only titles with more than one entry
placeholder = placeholder[placeholder['Global_Sales'] != 0.0]
plist = placeholder['Name'].to_list()

title_list = [] #A list of dataframes, with each list element being for one game title

for i in range(len(plist)):
    temp_df = placeholder[placeholder['Name'] == plist[i]]
    title_list.append(temp_df)

In [359]:
def title_gs_calc(title_df):
    """Takes a dataframe with multiple rows for a title and returns a dataframe with a single row with the Global_Sales calculated."""
    return_df = title_df.head(1).reset_index()
    return_df['Platform'] = 'All'
    return_df['Global_Sales'] = title_df['Global_Sales'].sum()    
    return return_df

In [360]:
def title_add(tlist):
    """Takes a list of dataframes consisting of the same title and returns a dataframe with All for the Platform value and the sum of the columns for Global_Sales."""
    added_titles = pd.DataFrame()
    for i in range(len(tlist)):
        temp_df = title_gs_calc(tlist[i])
        added_titles = pd.concat([added_titles, temp_df], ignore_index=True)
        added_titles = added_titles.drop_duplicates()     
    return added_titles

In [361]:
sales_data = pd.concat([sales_data, title_add(title_list)], ignore_index=True)
sales_data = sales_data.reset_index(drop=True)

remove_title_list = duplist(sales_data)
rows_to_remove = sales_data[sales_data['Name'].isin(remove_title_list)].index.to_list()
rows_to_save = sales_data[sales_data['Platform'] == 'All']

sales_data = sales_data.drop(rows_to_remove)
sales_data = pd.concat([sales_data, rows_to_save], ignore_index=True)
sales_data = sales_data.drop_duplicates()
sales_data = sales_data.drop(columns=sales_data.columns[5], axis=1)
print(sales_data.tail()) #As the new rows should be added to the bottom of the dataframe, we want to check the tail rather than the head.

                    Name Platform  Global_Sales    Year      Genre
13966  Iwaihime: Matsuri      All          0.01  2017.0  Adventure
13967   Closed Nightmare      All          0.01  2018.0       Misc
13968          Teslagrad      All          0.01  2014.0   Platform
13969             Island      All          0.01  2017.0  Adventure
13970     Super Meat Boy      All          0.01  2016.0   Platform


In [362]:
#Create list from Name column of top_50_data to use for column name for new dataframe created from Tags column
top_50_names = top_50_data['Name'].tolist()
top_50_data.index = top_50_names
top_50_data['Singleplayer'] = False
top_50_data['Multiplayer'] = False

In [363]:
#Utility function to check whether an element exists in a list of strings.

def list_check(selement: str, lelement: list[str]):
    """For a dataframe column consisting of lists of strings, return True if the tag exists in the list and False if it does not."""
    return selement in lelement

#Find Singleplayer and Multiplayer tags and set value of corresponding column to True if found

for name in top_50_names:
    taglist = top_50_data.loc[name, 'Tags']
    if list_check('Singleplayer', taglist):
        top_50_data.loc[name, 'Singleplayer'] = True
    else:
        top_50_data.loc[name, 'Singleplayer'] = False    
    if list_check('Multiplayer', taglist):
        top_50_data.loc[name, 'Multiplayer'] = True
    else:
        top_50_data.loc[name, 'Multiplayer'] = False

#Now that we have the Singleplayer and Multiplayer tags extracted we can drop the Tags column
top_50_data = top_50_data.drop(columns='Tags')

print(top_50_data.head())

                                                              Name       Year  \
Black Myth: Wukong                              Black Myth: Wukong  8/19/2024   
HELLDIVERS 2                                          HELLDIVERS 2   2/8/2024   
Palworld                                                  Palworld  1/18/2024   
Warhammer 40,000: Space Marine 2  Warhammer 40,000: Space Marine 2   9/9/2024   
Path of Exile 2                                    Path of Exile 2  12/6/2024   

                                   Price    Copies Sold    Gross Revenue  \
Black Myth: Wukong                $59.99  21,836,817.00  $951,505,160.00   
HELLDIVERS 2                      $39.99  12,410,685.00  $443,622,746.00   
Palworld                          $29.99  18,567,241.00  $435,465,541.00   
Warhammer 40,000: Space Marine 2  $59.99   2,709,725.00  $142,612,930.00   
Path of Exile 2                   $29.99   5,015,358.00  $135,369,527.00   

                                  Average Playtime (Hrs)

In [364]:
#Get column averages for specified and append to new columns at end

gmask = gaming_study_data['Name'].unique().tolist()

get_average(gaming_study_data, gmask, 'GAD_Total')
get_average(gaming_study_data, gmask, 'SWL_Total')
get_average(gaming_study_data, gmask, 'SPIN_Total')
get_average(gaming_study_data, gmask, 'HoursPerWeek')

#Create new dataframe with one entry per game.  The individual item averages for each screening test are not included in the dataframe
# as it is intended to be a high-level overview.

gaming_study_avg = gaming_study_data.filter(['Name', 'HoursPerWeek_avg', 'GAD_Total_avg', 'SWL_Total_avg', 'SPIN_Total_avg'], axis=1)
gaming_study_avg = gaming_study_avg.drop_duplicates()

In [365]:
#Open the SQL connection and create a cursor

conn = sqlite3.connect(':memory:') 
cursor = conn.cursor()

def sql(query):
    """Input a SQL query and return a pandas dataframe object."""
    return pd.read_sql_query(query, conn)

#Load our data into SQL tables

sales_data.to_sql('Sales', conn, if_exists='replace', index=False)
gaming_study_avg.to_sql('Gaming_Study_Avg', conn, if_exists='replace', index=False)

conn.execute("""
    CREATE TABLE SalesT
    (
    Name TEXT PRIMARY KEY,
    Platform TEXT,
    Global_Sales REAL,
    Genre TEXT,
    Year INTEGER);
    """)
    
conn.execute("""
    INSERT OR REPLACE INTO SalesT (Name, Platform, Global_Sales, Genre, Year)
    SELECT Name, Platform, Global_Sales, Genre, Year FROM Sales;
    """)

conn.execute("""
    CREATE TABLE GST
    (
    Name TEXT PRIMARY KEY,
    HoursPerWeek_avg REAL,
    GAD_Total_avg REAL,
    SWL_Total_avg REAL,
    SPIN_Total_avg REAL);
    """)

conn.execute("""
    INSERT OR REPLACE INTO GST (Name, HoursPerWeek_avg, GAD_Total_avg, SWL_Total_avg, SPIN_Total_avg)
    SELECT Name, HoursPerWeek_avg, GAD_Total_avg, SWL_Total_avg, SPIN_Total_avg FROM Gaming_Study_Avg;
    """)

sales_and_study = sql("""
    SELECT *
    FROM GST
    LEFT JOIN SalesT ON SalesT.Name = GST.Name;
    """)
sales_and_study = sales_and_study.loc[:,~sales_and_study.columns.duplicated()]

conn.close()


In [366]:
#Export our updated and new dataframes to CSV to be loaded into Tableau!

sales_data.to_csv('Final_Data/Sales.csv', index=False)
top_50_data.to_csv('Final_Data/Top_50.csv', index=False)
gaming_study_data.to_csv('Final_Data/Gaming_Study.csv', index=False)
online_behavior_data.to_csv('Final_Data/Behavior.csv', index=False)
sales_and_study.to_csv('Final_Data/Sales_and_Study.csv', index=False)